In [4]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score

In [5]:
# generate dataset
X, y = make_moons(n_samples=10000, noise=0.4, random_state=42)

# split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

Train size: 8000, Test size: 2000


In [ ]:
# define individual classifiers
lr = LogisticRegression(random_state=42)
svm = SVC(probability=True, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# combine them in a voting classifier
voting_clf = VotingClassifier(
    estimators=[('lr', lr), ('svm', svm), ('rf', rf)],
    voting='soft'
)

classifiers = {
    'LogisticRegression': lr,
    'SVM': svm,
    'RandomForest': rf,
    'VotingClassifier': voting_clf
}

In [ ]:
# train and evaluate each classifier
print("Training and evaluation results:")
print("-" * 45)

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)

    train_pred = clf.predict(X_train)
    test_pred = clf.predict(X_test)

    train_err = 1 - accuracy_score(y_train, train_pred)
    test_err = 1 - accuracy_score(y_test, test_pred)

    print(f"{name}:")
    print(f"  train error: {train_err:.4f}  |  test error: {test_err:.4f}")

print("-" * 45)

In [ ]:
# plot class boundaries for the voting classifier
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))

Z = voting_clf.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, s=8, cmap='coolwarm', edgecolors='k', linewidths=0.2)
plt.title("VotingClassifier - Class Boundaries (test set)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.tight_layout()
plt.show()